# Sentinel-2 L1C Dataset Builder
Exports unmasked B08, B04, B03 and B02 GeoTIFFs for the CloudSEN12 `dtacs4bands` model. Scene-wide cloud metadata is used only to create a diverse test set; it is not ground truth for evaluation.

In [ ]:
!pip -q install earthengine-api geemap pandas
import ee
import pandas as pd
PROJECT_ID = 'REPLACE_WITH_YOUR_GOOGLE_CLOUD_PROJECT'
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
# Four agricultural regions for geographically diverse operational tests.
AOIS = {
    'bulgaria_sofia_basin': [23.05, 42.55, 23.35, 42.75],
    'italy_po_valley': [10.45, 44.70, 10.75, 44.90],
    'spain_castilla': [-4.90, 41.50, -4.60, 41.70],
    'romania_danube_plain': [25.40, 44.00, 25.70, 44.20],
}
START_DATE = '2025-03-01'
END_DATE = '2025-10-31'
TARGET_CLOUD_LEVELS = [10, 40, 80]

In [ ]:
selected = []
for region_name, bounds in AOIS.items():
    aoi = ee.Geometry.Rectangle(bounds)
    collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(START_DATE, END_DATE))
    for target in TARGET_CLOUD_LEVELS:
        candidates = (collection
            .filter(ee.Filter.gte('CLOUDY_PIXEL_PERCENTAGE', target - 10))
            .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', target + 10))
            .sort('CLOUDY_PIXEL_PERCENTAGE'))
        if candidates.size().getInfo() == 0:
            continue
        image = candidates.first()
        selected.append({
            'region': region_name,
            'aoi': aoi,
            'image': image,
            'product_id': image.get('PRODUCT_ID').getInfo(),
            'cloud_metadata': image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
        })
pd.DataFrame([{k:v for k,v in item.items() if k not in ('aoi','image')} for item in selected])

In [ ]:
# Start Google Drive exports. No cloud mask is applied.
tasks = []
for index, item in enumerate(selected, start=1):
    export_image = item['image'].select(
        ['B8', 'B4', 'B3', 'B2'],
        ['B08', 'B04', 'B03', 'B02'],
    ).toUint16()
    prefix = f"cloud_scene_{index:02d}_{item['region']}"
    task = ee.batch.Export.image.toDrive(
        image=export_image,
        description=prefix,
        folder='ViTA_CloudDetection',
        fileNamePrefix=prefix,
        region=item['aoi'],
        scale=10,
        maxPixels=1e9,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
    )
    task.start()
    tasks.append({'file': prefix, 'task_id': task.id, 'product_id': item['product_id']})
pd.DataFrame(tasks)